# Day 2-04｜多個球員框投影到鳥瞰圖
> Python 籃球運動資料分析課程  
> 整合 Day 1 的 Homography 與 Day 2 的 Detection，將多個球員位置轉成 BEV 點位。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 讀取多個 player detections。
- 為每個 bbox 計算 bottom-center footpoint。
- 把所有球員位置投影到 BEV。

## 完成產出
- 相機畫面與 BEV 球員位置的整合視覺化結果。

## 課堂要求
- 按照本單元順序執行各段程式。
- 僅修改題目指定的變數、路徑或參數。
- 完成指定輸出後，記錄結果並供課堂討論。


## 課程流程
1. 載入 detection 與 Homography。
2. 篩選 player 類別並計算 footpoint。
3. 投影並儲存整合圖。


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


In [ ]:
from src.cv_utils import (
    read_image_rgb,
    load_json,
    draw_boxes,
    draw_points,
    show_image,
    side_by_side,
    bottom_center,
    save_image_rgb,
)
from src.geometry_utils import compute_homography, project_points, render_bev_court

image = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png")
bev = render_bev_court(COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json")
homo = load_json(COURSE_ROOT / "assets" / "samples" / "sample_homography_points.json")
H = compute_homography(homo["camera_points"], homo["bev_points"])

data = load_json(COURSE_ROOT / "assets" / "samples" / "sample_detections_frame0.json")
players = [
    d
    for d in data["detections"]
    if d["class_name"] == "player" and d["confidence"] >= 0.75
]

boxes = [d["bbox_xyxy"] for d in players]
feet = [bottom_center(b) for b in boxes]
bev_feet = project_points(feet, H)

print("players:", len(players))

In [ ]:
labels = [f"p{i}" for i in range(len(players))]
frame_vis = draw_boxes(image, boxes, labels)
frame_vis = draw_points(frame_vis, feet, labels)
bev_vis = draw_points(bev, bev_feet, labels)
combo = side_by_side(frame_vis, bev_vis)
show_image(combo, "multi-player BBOX-to-BEV")

save_path = COURSE_ROOT / "assets" / "results" / "d2_04_bbox_to_bev_integration.png"
save_image_rgb(save_path, combo)
print("saved:", save_path)

本單元暫不處理 track ID；下一單元將討論同一名球員在不同 frame 中如何維持一致 ID。